# 12. Генеративные модели: VAE, GAN, диффузия, LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/12_generative.ipynb)

Код из раздела [modules/12_generative.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/12_generative.md#m12). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### VAE и ELBO

In [ ]:
def vae_loss(x, x_rec, mu, logvar):
    rec = F.mse_loss(x_rec, x, reduction="sum")                        # −log p(x|z) для гауссова декодера
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())      # формула из модуля 08
    return rec + kl
# сэмплирование z с репараметризацией: z = mu + exp(0.5*logvar) * randn

### 🏋️ Практика модуля

**12.1.** Реализуйте top-p сэмплирование.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
def top_p_sample(logits, p=0.9, T=1.0, rng=np.random.default_rng()):
    z = logits / T
    probs = np.exp(z - z.max()); probs /= probs.sum()
    order = np.argsort(-probs)
    cum = np.cumsum(probs[order])
    keep = order[: np.searchsorted(cum, p) + 1]          # минимальный набор с массой ≥ p
    q = probs[keep] / probs[keep].sum()
    return rng.choice(keep, p=q)

### 🏋️ Практика модуля

**12.2.** Для линейного расписания $\beta_t$ от $10^{-4}$ до $0.02$, $T = 1000$ найдите долю сигнала $\sqrt{\bar\alpha_t}$ при $t = 500$ и $t = 1000$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
betas = np.linspace(1e-4, 0.02, 1000); abar = np.cumprod(1 - betas)
print(np.sqrt(abar[499]), np.sqrt(abar[-1]))   # ≈ 0.28 и ≈ 0.006

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def diffusion():
    T = 1000
    betas = np.linspace(1e-4, 0.02, T)
    abar = np.cumprod(1 - betas)
    s = 0.008; tt = np.arange(T + 1) / T
    f = np.cos((tt + s) / (1 + s) * np.pi / 2) ** 2
    abar_cos = (f / f[0])[1:]
    yy, xx = np.mgrid[-1:1:64j, -1:1:64j]
    x0 = ((xx**2 + yy**2 < 0.5) & (np.abs(xx) + np.abs(yy) > 0.45)).astype(float) * 2 - 1
    rng = np.random.default_rng(0); eps = rng.normal(size=x0.shape)
    fig = plt.figure(figsize=(15, 4.2))
    steps = [0, 50, 150, 300, 600, 999]
    for i, t in enumerate(steps):
        ax = fig.add_subplot(2, 6, i + 1)
        xt = np.sqrt(abar[t]) * x0 + np.sqrt(1 - abar[t]) * eps
        ax.imshow(xt, cmap="gray"); ax.axis("off"); ax.set_title(f"t={t}", fontsize=9)
    ax = fig.add_subplot(2, 1, 2)
    ax.plot(abar, color=C[0], lw=2, label="ᾱₜ линейное расписание β")
    ax.plot(abar_cos, color=C[1], lw=2, label="ᾱₜ косинусное расписание")
    ax.set_xlabel("шаг t"); ax.set_ylabel("доля сигнала ᾱₜ"); ax.legend()
    fig.suptitle("Диффузия: xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε. Модель учится предсказывать шум ε и идёт обратно", y=1.0)
    save(fig, "diffusion")

diffusion()